In [4]:
import pynucastro as pyna
import numpy as np
from pathlib import Path

In [5]:
def parse_rate_line(line):
    # Extract fields by fixed positions
    nuclides = [line[5 + i*5 : 10 + i*5].strip() for i in range(6)]
    set_label = line[43:47].strip()
    rate_flag = line[47].strip()  # 'n', 'r', or 'w'
    reverse_flag = line[48].strip()  # 'v' if reverse
    q_value = float(line[52:64].strip())
    return {
        "nuclides": nuclides,
        "set_label": set_label,
        "rate_flag": rate_flag,
        "reverse_flag": reverse_flag,
        "q_value": q_value
    }

def is_alpha_decay(line):
    """Check if Reaclib header line corresponds to an alpha decay."""
   
    data= parse_rate_line(line)
    nuclides = data['nuclides']
    nuclides = [n for n in nuclides if n]
    if nuclides[0]=='al-6' or nuclides[0]=='al*6' or nuclides[2]=='al-6' or nuclides[2]=='al*6':
        return False
    else:
        flag=pyna.nucdata.nucleus.Nucleus(nuclides[0]).Z==pyna.nucdata.nucleus.Nucleus(nuclides[2]).Z+2
        if flag and data['q_value']>0 and nuclides[1]=='he4':
            return True


def zero_coefficients_line(line1,line2):
    coeffients1=[float(line1[i*13:+13+i*13].strip()) for i in range(4)]
    coeffients2=[float(line2[i*13:+13+i*13].strip()) for i in range(2)]
   
    new=sum(np.array(coeffients1))+sum(np.array(coeffients2))
    """Replace all coefficients (scientific notation values) with 0.00000e+00."""
    if new<0:
        return f"{new:.6e}"+" 0.000000e+00"*3 + " "*18+'\n'
    else:
        return f" {new:.6e}"+" 0.000000e+00"*3 + " "*18+'\n'
    

def process_reaclib(infile, outfile):
    """Reads a Reaclib file and zeros out coefficients for alpha decays."""
    with open(infile, "r") as fin, open(outfile, "w") as fout:
        lines = fin.readlines()
        i = 0
        chapter=False
        while i < len(lines):
            line = lines[i]
            if line[0]=='2':
                chapter=True
            elif line[0]!=' ' and line[0]!='-':
                chapter=False
            if chapter and line[0:5] == "     " and line[0:10]!='          ':
                
                # This is a header line
                fout.write(line)
                if is_alpha_decay(line):
                    
                    if i + 3 < len(lines):
                        fout.write(zero_coefficients_line(lines[i + 1],lines[i+2]))
                        fout.write(" 0.000000e+00"*3 + " "*33+'\n')
                    i += 3
                    continue
                
            
            else:
                fout.write(line)
            i += 1

In [6]:
output_file=Path(r"C:\users\Diego Hernandez\Documents\GODIE_DOCUMENTOS\importante\Kilonova_3.0\Nuclear_Data\run4\Reaclib_NO_T") 
input_file=Path(r"C:\users\Diego Hernandez\Documents\GODIE_DOCUMENTOS\importante\Kilonova_3.0\Nuclear_Data\run1\Reaclib_18_9_20")
process_reaclib(input_file,output_file)